In [4]:
%pip install numpy
%pip install pandas
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ------------------------- -------------- 5.2/8.3 MB 33.4 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 29.5 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   -------- ------------------------------- 8.4/37.4 MB 41.2 MB/s eta 0:00:01
   ------------------ --------------------- 17.3/37.4 MB 41.9 MB/s eta 0:00:01
   ---------------------------- ----------- 26.2/37.4 MB 41.8 MB/s eta 0:00:01
   ------------------------------------ --- 33.8/37.4 MB 40.7 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 41.3 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 35.8 MB/s  0:00:01

   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ---


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%wget -r -N -c -np https://physionet.org/files/bidmc/1.0.0/bidmc_csv/ -A "*_Numerics.csv"

In [2]:
import glob
import os
import re
import pandas as pd

# Path pattern to find all Numerics CSV files under the directory structure
pattern = os.path.join(
    "physionet.org",
    "files",
    "bidmc",
    "1.0.0",
    "bidmc_csv",
    "bidmc_*_Numerics.csv",
)

files = sorted(glob.glob(pattern))
print(files)
dfs = []

for file_path in files:
    df = pd.read_csv(file_path)

    # Extract subject ID (e.g., '01' from 'bidmc_01_Numerics.csv')
    match = re.search(r"bidmc_(\d+)_Numerics\.csv", file_path)
    if match:
        df["subject_id"] = match.group(1)

    dfs.append(df)

# Combine all DataFrames and export to top-level bidmc.csv
combined_df = pd.concat(dfs, ignore_index=True)
combined_df.to_csv("bidmc.csv", index=False)

print(f"Successfully combined {len(files)} files into bidmc.csv")

['physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_01_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_02_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_03_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_04_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_05_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_06_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_07_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_08_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_09_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_10_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_11_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_12_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_13_Numerics.csv', 'physionet.org\\files\\bidmc\\1.0.0\\bidmc_csv\\bidmc_14_Numeri

In [ ]:
import numpy as np
import pandas as pd

VITALS = ["HR", "PULSE", "RESP", "SpO2"]
SLOPE_WINDOW = 30
BASELINE_MIN_PERIODS = 60

df = pd.read_csv("data/bidmc.csv")
df.columns = df.columns.str.strip()
df["subject_id"] = df["subject_id"].astype(str)
df = df.sort_values(["subject_id", "Time [s]"]).reset_index(drop=True)

['Time [s]', 'HR', 'PULSE', 'RESP', 'SpO2', 'subject_id']

In [ ]:
def rolling_slope(series: pd.Series, window: int) -> pd.Series:
    x = np.arange(window)
    x_mean = x.mean()
    denom = ((x - x_mean) ** 2).sum()

    def _slope(y):
        y_mean = y.mean()
        return ((x - x_mean) * (y - y_mean)).sum() / denom

    return series.rolling(window=window, min_periods=window).apply(_slope, raw=True)


def engineer_telemetry_features(group: pd.DataFrame) -> pd.DataFrame:
    """Feature-engineer a single subject's time series. Returns same # of rows as input."""
    group = group.copy()

    # Personal-baseline z-scores
    for vital in VITALS:
        mean = group[vital].mean()
        std = group[vital].std() or np.nan
        group[f"{vital}_zscore"] = (group[vital] - mean) / std

    # Rolling slopes (rate of change)
    for vital in VITALS:
        group[f"{vital}_slope"] = rolling_slope(group[vital], SLOPE_WINDOW)

    # Cross-vital interactions
    group["HR_SpO2_interaction"] = group["HR_zscore"] * group["SpO2_zscore"]
    group["HR_RESP_interaction"] = group["HR_zscore"] * group["RESP_zscore"]
    group["RESP_SpO2_interaction"] = group["RESP_zscore"] * group["SpO2_zscore"]
    group["HR_PULSE_divergence"] = (group["HR_zscore"] - group["PULSE_zscore"]).abs()

    # Regime proxy (heuristic stand-in — BIDMC has no real activity label)
    hr_rolling_std = group["HR"].rolling(window=SLOPE_WINDOW, min_periods=SLOPE_WINDOW).std()
    group["regime_proxy"] = pd.cut(
        hr_rolling_std,
        bins=[-np.inf, 1.0, 3.0, np.inf],
        labels=["low_variability", "medium_variability", "high_variability"],
    )

    return group

In [11]:
df_featured = pd.concat(
    [engineer_telemetry_features(group) for _, group in df.groupby("subject_id")],
    ignore_index=True,
)

# Drop leading rows per subject where baseline/rolling stats aren't reliable yet
valid_counts = df_featured.groupby("subject_id").cumcount() + 1
df_featured = df_featured[valid_counts >= BASELINE_MIN_PERIODS].reset_index(drop=True)

print(df_featured.shape)
df_featured.head()

KeyError: 'HR'